In [0]:
import boto3
AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""

AWS_BUCKET_NAME = ""

s3 = boto3.client(
    "s3",
    aws_access_key_id="",
    aws_secret_access_key=""
)

In [0]:
from pyspark.sql.functions import *

In [0]:
silver_df=spark.table("silver_youtube_events")

In [0]:
silver_df.show(5, truncate=False)

+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+--------------------------+----------------+
|country|device_type|event_time                |liked|subscribed|user_id|video_category|video_id|video_language|watch_time|ingestion_time            |engagement_score|
+-------+-----------+--------------------------+-----+----------+-------+--------------+--------+--------------+----------+--------------------------+----------------+
|USA    |Desktop    |2026-05-28 09:20:53.245397|true |false     |4637   |Tech          |vid_3   |Hindi         |302       |2026-05-28 09:24:19.259333|1               |
|Germany|Desktop    |2026-05-28 09:20:53.24556 |false|false     |5534   |Vlogs         |vid_21  |Hindi         |798       |2026-05-28 09:24:19.259333|0               |
|UK     |Mobile     |2026-05-28 09:20:53.245576|true |true      |5413   |Gaming        |vid_30  |Spanish       |1109      |2026-05-28 09:24:19.259333|3         

In [0]:
category_kpis = silver_df.groupBy("video_category") \
    .agg(
        count("*").alias("total_views"),
        avg("watch_time").alias("avg_watch_time"),
        sum("engagement_score").alias("total_engagement")
    )

In [0]:
category_kpis.show(truncate=False)

+--------------+-----------+-----------------+----------------+
|video_category|total_views|avg_watch_time   |total_engagement|
+--------------+-----------+-----------------+----------------+
|Tech          |193        |613.5595854922279|277             |
|Vlogs         |215        |582.0790697674419|339             |
|Gaming        |196        |606.7602040816327|300             |
|Education     |195        |600.7435897435897|279             |
|AI            |201        |587.363184079602 |294             |
+--------------+-----------+-----------------+----------------+



In [0]:
country_kpis = silver_df.groupBy("country") \
    .agg(
        count("*").alias("total_views"),
        avg("watch_time").alias("avg_watch_time"),
        sum("engagement_score").alias("total_engagement")
    )

In [0]:
country_kpis.show(truncate=False)

+-------+-----------+-----------------+----------------+
|country|total_views|avg_watch_time   |total_engagement|
+-------+-----------+-----------------+----------------+
|USA    |201        |596.8308457711443|274             |
|Germany|203        |553.5369458128079|294             |
|UK     |193        |660.3212435233161|285             |
|India  |201        |574.1343283582089|308             |
|Canada |202        |606.5346534653465|328             |
+-------+-----------+-----------------+----------------+



In [0]:
device_kpis = silver_df.groupBy("device_type") \
    .agg(
        count("*").alias("total_views"),
        avg("watch_time").alias("avg_watch_time")
    )

In [0]:
device_kpis.show(truncate=False)

+-----------+-----------+-----------------+
|device_type|total_views|avg_watch_time   |
+-----------+-----------+-----------------+
|Desktop    |314        |611.2133757961783|
|Mobile     |337        |576.9762611275964|
|Tablet     |349        |605.5358166189112|
+-----------+-----------+-----------------+



In [0]:
category_kpis.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_category_kpis")

In [0]:
country_kpis.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_country_kpis")

In [0]:
device_kpis.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_device_kpis")

In [0]:
spark.sql("""
SELECT *
FROM gold_category_kpis
""").show(truncate=False)

+--------------+-----------+-----------------+----------------+
|video_category|total_views|avg_watch_time   |total_engagement|
+--------------+-----------+-----------------+----------------+
|Vlogs         |215        |582.0790697674419|339             |
|Gaming        |196        |606.7602040816327|300             |
|AI            |201        |587.363184079602 |294             |
|Education     |195        |600.7435897435897|279             |
|Tech          |193        |613.5595854922279|277             |
+--------------+-----------+-----------------+----------------+



In [0]:
spark.sql("""
SELECT *
FROM gold_country_kpis
""").show(truncate=False)

+-------+-----------+-----------------+----------------+
|country|total_views|avg_watch_time   |total_engagement|
+-------+-----------+-----------------+----------------+
|USA    |201        |596.8308457711443|274             |
|Canada |202        |606.5346534653465|328             |
|India  |201        |574.1343283582089|308             |
|UK     |193        |660.3212435233161|285             |
|Germany|203        |553.5369458128079|294             |
+-------+-----------+-----------------+----------------+



In [0]:
spark.sql("""
SELECT *
FROM gold_device_kpis
""").show(truncate=False)

+-----------+-----------+-----------------+
|device_type|total_views|avg_watch_time   |
+-----------+-----------+-----------------+
|Tablet     |349        |605.5358166189112|
|Desktop    |314        |611.2133757961783|
|Mobile     |337        |576.9762611275964|
+-----------+-----------+-----------------+



In [0]:
category_export = category_kpis.toPandas()

In [0]:
category_export.to_csv(
    "/tmp/gold_category_kpis.csv",
    index=False
)

In [0]:
country_export = country_kpis.toPandas()

In [0]:
country_export.to_csv(
    "/tmp/gold_country_kpis.csv",
    index=False
)

In [0]:
s3.upload_file(
    "/tmp/gold_country_kpis.csv",
    "youtube-analytics-lakehouse",
    "gold/country/gold_country_kpis.csv"
)

print("Country KPIs uploaded")

Country KPIs uploaded


In [0]:
device_export = device_kpis.toPandas()

In [0]:
device_export.to_csv(
    "/tmp/gold_device_kpis.csv",
    index=False
)

In [0]:
s3.upload_file(
    "/tmp/gold_device_kpis.csv",
    "youtube-analytics-lakehouse",
    "gold/device/gold_device_kpis.csv"
)

print("Device KPIs uploaded")

Device KPIs uploaded


In [0]:
s3.upload_file(
    "/tmp/gold_category_kpis.csv",
    "youtube-analytics-lakehouse",
    "gold/category/gold_category_kpis.csv"
)

print("Category KPIs uploaded")

Category KPIs uploaded
